In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

input_dir = Path("/kaggle/input/datasets/emmagasparato/eurocrops")
output_dir = Path("/kaggle/working/eurocrops_csv")
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
for file in input_dir.rglob("*.parquet"):
    df = gpd.read_parquet(file)

    # Convert the geometrr in WKT text
    if isinstance(df, gpd.GeoDataFrame):
        df = df.to_crs("EPSG:4326")
        df["geometry"] = df.geometry.to_wkt()
        df = pd.DataFrame(df)

    output_file = output_dir / f"{file.stem}.csv"
    df.to_csv(output_file, index=False)

    print(f"Converted: {file.name} -> {output_file.name}")

/tmp/ipykernel_16/2817965703.py:7: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


Converted: iti1_2020.parquet -> iti1_2020.csv


/tmp/ipykernel_16/2817965703.py:7: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


Converted: iti1_2021.parquet -> iti1_2021.csv


/tmp/ipykernel_16/2817965703.py:7: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


Converted: iti1_2022.parquet -> iti1_2022.csv


/tmp/ipykernel_16/2817965703.py:7: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.geometry.to_wkt()


Converted: iti1_2023.parquet -> iti1_2023.csv


In [3]:
crop_rotations_2020 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2020.csv")
crop_rotations_2021 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2021.csv")
crop_rotations_2022 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2022.csv")
crop_rotations_2023 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2023.csv")

In [4]:
mapping = pd.read_csv("/kaggle/input/datasets/emmagasparato/eurocrops/eurocrops.csv")

print(mapping.columns.tolist())
display(mapping.head())

mapping = mapping[mapping["nuts"].str.lower() == "iti1"].copy()
print("Crops in the mapping:", len(mapping))

crop_rotations_2020["original_code"] = crop_rotations_2020["original_code"].astype(str)
mapping["original_code"] = mapping["original_code"].astype(str)

crop_rotations_2020 = crop_rotations_2020.merge(
    mapping[["original_code", "original_name", "translated_name", "hcat4_name"]].drop_duplicates(),
    on="original_code",
    how="left",
    validate="many_to_one"
)

display(crop_rotations_2020.head())

['nuts', 'original_code', 'original_name', 'translated_name', 'hcat4_code', 'hcat4_name', 'usage_code', 'usage_name']


,nuts,original_code,original_name,translated_name,hcat4_code,hcat4_name,usage_code,usage_name
0,at,10001,NICHT DEFINIERT,not defined,3390000000,not_known_and_other,0,No specified use
1,at,105,KÖRNERMAIS,GRAIN CORN,3310106000,maize_corn_popcorn,6,Harvested as Grains
2,at,106,MAIS CORN-COB-MIX (CCM),CORN CORN-COB-MIX (CCM),3310106000,maize_corn_popcorn,0,No specified use
3,at,107,ZUCKERMAIS,SUGAR CORN,3310106000,maize_corn_popcorn,0,No specified use
4,at,109,SILOMAIS,SILO CORN,3310106000,maize_corn_popcorn,3,Fodder/Forage


Crops in the mapping: 369


,cropfield,original_code,off_id,off_area,area_ha,geometry,original_name,translated_name,hcat4_name
0,671471,7001,459144,1,0.000135,"MULTIPOLYGON (((11.489213 43.255067, 11.489221...",Fabbricati,Buildings,not_known_and_other
1,671805,2302,461788,5124,0.512238,"MULTIPOLYGON (((11.573557 43.312212, 11.572889...",VITE,GRAPE,vineyards_wine_vine_rebland_grapes
2,92043,1095,535195,34006,0.000203,"MULTIPOLYGON (((12.108821 43.295675, 12.108821...",Bosco,Woodland,tree_wood_forest
3,549588,336,704421,310,0.030981,"MULTIPOLYGON (((10.884328 43.63673, 10.884253 ...",PRATO POLIFITA,Multifloral Meadow,permanent_grassland
4,549650,666,705152,15,0.001453,"MULTIPOLYGON (((10.72121 43.758108, 10.7211 43...",SEMINATIVI,ARABLE LAND,arable_crops


In [5]:
print("Crops without correspondence:",
      crop_rotations_2020["hcat4_name"].isna().sum())

display(
    crop_rotations_2020[
        ["cropfield", "original_code", "translated_name", "hcat4_name", "area_ha"]
    ].head(10)
)

Crops without correspondence: 0


,cropfield,original_code,translated_name,hcat4_name,area_ha
0,671471,7001,Buildings,not_known_and_other,0.000135
1,671805,2302,GRAPE,vineyards_wine_vine_rebland_grapes,0.512238
2,92043,1095,Woodland,tree_wood_forest,0.000203
3,549588,336,Multifloral Meadow,permanent_grassland,0.030981
4,549650,666,ARABLE LAND,arable_crops,0.001453
5,549721,2302,GRAPE,vineyards_wine_vine_rebland_grapes,1.382892
6,293151,1700,Land withdrawn from production,unmaintained,0.000112
7,290426,2402,Olive for processing,olive,0.928742
8,322014,666,ARABLE LAND,arable_crops,1.840553
9,328269,1700,Land withdrawn from production,unmaintained,0.142265


In [6]:
crop_areas = (
    crop_rotations_2020.groupby("hcat4_name", dropna=False)["area_ha"]
    .sum()
    .sort_values(ascending=False)
)

display(crop_areas.head(20))

hcat4_name
tree_wood_forest                      114866.142896
arable_crops                           76143.201121
olive                                  62273.613402
poaceae_grasses                        58810.111738
vineyards_wine_vine_rebland_grapes     56569.494225
unmaintained                           54810.053946
permanent_grassland                    53298.573229
alfalfa_lucerne                        48624.901206
durum_hard_wheat                       45580.733131
common_soft_wheat                      22405.050432
clover                                 17738.071057
barley                                 16992.739414
sunflower                              15406.077919
beans                                  10672.502826
maize_corn_popcorn                      9430.655731
oats                                    9217.525259
spelt                                   6638.786773
sorghum                                 3891.291068
nurseries_nursery                       3510.241079
n

In [7]:
mapping["original_code"] = mapping["original_code"].astype(str)

crop_rotations = {2020: crop_rotations_2020}

for year in [2021, 2022, 2023]:
    df = pd.read_csv(f"/kaggle/working/eurocrops_csv/iti1_{year}.csv")
    df["original_code"] = df["original_code"].astype(str)

    df = df.merge(
        mapping[["original_code", "original_name", "translated_name", "hcat4_name"]].drop_duplicates(),
        on="original_code",
        how="left",
        validate="many_to_one"
    )

    crop_rotations[year] = df
    print(f"{year}: {len(df)} lots, {df['hcat4_name'].isna().sum()} without mapping")

for year in [2021, 2022, 2023]:
    crop_rotations[year].to_csv(
        f"/kaggle/working/iti1_{year}_mapped.csv", index=False
    )

2021: 671747 lots, 0 without mapping
2022: 668495 lots, 0 without mapping
2023: 580286 lots, 0 without mapping


In [8]:
crop_rotations_all = pd.concat([
    crop_rotations_2020.assign(year=2020),
    crop_rotations_2021.assign(year=2021),
    crop_rotations_2022.assign(year=2022),
    crop_rotations_2023.assign(year=2023)
], ignore_index=True)

display(crop_rotations_all.head())
print("Dimensions:", crop_rotations_all.shape)

,cropfield,original_code,off_id,off_area,area_ha,geometry,original_name,translated_name,hcat4_name,year
0,671471,7001,459144,1,0.000135,"MULTIPOLYGON (((11.489213 43.255067, 11.489221...",Fabbricati,Buildings,not_known_and_other,2020
1,671805,2302,461788,5124,0.512238,"MULTIPOLYGON (((11.573557 43.312212, 11.572889...",VITE,GRAPE,vineyards_wine_vine_rebland_grapes,2020
2,92043,1095,535195,34006,0.000203,"MULTIPOLYGON (((12.108821 43.295675, 12.108821...",Bosco,Woodland,tree_wood_forest,2020
3,549588,336,704421,310,0.030981,"MULTIPOLYGON (((10.884328 43.63673, 10.884253 ...",PRATO POLIFITA,Multifloral Meadow,permanent_grassland,2020
4,549650,666,705152,15,0.001453,"MULTIPOLYGON (((10.72121 43.758108, 10.7211 43...",SEMINATIVI,ARABLE LAND,arable_crops,2020


Dimensions: (2592275, 10)


In [9]:
crop_rotations_all.to_csv(
    "/kaggle/working/eurocrops_toscana_2020_2023.csv",
    index=False
)
print(crop_rotations_all.groupby("year").size())
print(crop_rotations_all.isna().sum())

year
2020    671747
2021    671747
2022    668495
2023    580286
dtype: int64
cropfield                0
original_code            0
off_id                   0
off_area                 0
area_ha                  0
geometry                 0
original_name      1920528
translated_name    1920528
hcat4_name         1920528
year                     0
dtype: int64


In [10]:
crop_rotations_all.head()

,cropfield,original_code,off_id,off_area,area_ha,geometry,original_name,translated_name,hcat4_name,year
0,671471,7001,459144,1,0.000135,"MULTIPOLYGON (((11.489213 43.255067, 11.489221...",Fabbricati,Buildings,not_known_and_other,2020
1,671805,2302,461788,5124,0.512238,"MULTIPOLYGON (((11.573557 43.312212, 11.572889...",VITE,GRAPE,vineyards_wine_vine_rebland_grapes,2020
2,92043,1095,535195,34006,0.000203,"MULTIPOLYGON (((12.108821 43.295675, 12.108821...",Bosco,Woodland,tree_wood_forest,2020
3,549588,336,704421,310,0.030981,"MULTIPOLYGON (((10.884328 43.63673, 10.884253 ...",PRATO POLIFITA,Multifloral Meadow,permanent_grassland,2020
4,549650,666,705152,15,0.001453,"MULTIPOLYGON (((10.72121 43.758108, 10.7211 43...",SEMINATIVI,ARABLE LAND,arable_crops,2020


In [11]:
import geopandas as gpd

base = Path("/kaggle/input/datasets/emmagasparato/eurocrops")

mapping = pd.read_csv(base / "eurocrops.csv", dtype={"original_code": str})
mapping = mapping[mapping["nuts"].str.lower() == "iti1"]
mapping = mapping[["original_code", "hcat4_name"]].drop_duplicates("original_code")

fields = {}

for year in [2020, 2021, 2022, 2023]:
    gdf = gpd.read_parquet(base / f"iti1_{year}.parquet")
    gdf["original_code"] = gdf["original_code"].astype(str)
    gdf = gdf.merge(mapping, on="original_code", how="left")
    gdf = gdf.to_crs("EPSG:3035").reset_index(drop=True)
    gdf["field_id"] = gdf.index

    fields[year] = gdf
    print(year, len(gdf))

2020 671747
2021 671747
2022 668495
2023 580286


In [12]:
def match_fields(left, right, threshold=0.8):
    a = left[["field_id", "geometry"]].copy()
    b = right[["field_id", "geometry"]].copy()

    pairs = gpd.sjoin(a, b, how="inner", predicate="intersects",
                      lsuffix="left", rsuffix="right")

    if pairs.empty:
        return pd.DataFrame(columns=["id_left", "id_right", "overlap"])

    ids_a = pairs["field_id_left"].to_numpy()
    ids_b = pairs["field_id_right"].to_numpy()

    ga = a.geometry.iloc[ids_a].reset_index(drop=True)
    gb = b.geometry.iloc[ids_b].reset_index(drop=True)

    intersection = ga.intersection(gb).area.to_numpy()
    overlap = intersection / pd.concat([
        ga.area, gb.area
    ], axis=1).max(axis=1).to_numpy()

    matches = pd.DataFrame({
        "id_left": ids_a,
        "id_right": ids_b,
        "overlap": overlap
    })

    matches = matches[matches["overlap"] >= threshold]
    matches = matches.sort_values("overlap", ascending=False)
    matches = matches.drop_duplicates("id_left").drop_duplicates("id_right")

    return matches.reset_index(drop=True)

In [13]:
matches = {}

for year in [2020, 2021, 2022]:
    m = match_fields(fields[year], fields[year + 1])
    matches[year] = m
    print(f"{year} → {year+1}: {len(m)} associated fields")

2020 → 2021: 670773 associated fields
2021 → 2022: 534930 associated fields
2022 → 2023: 435554 associated fields


In [14]:
sequences = matches[2020][["id_left", "id_right"]].rename(
    columns={"id_left": "id_2020", "id_right": "id_2021"}
)

for year in [2021, 2022]:
    m = matches[year][["id_left", "id_right"]].rename(
        columns={"id_left": f"id_{year}", "id_right": f"id_{year+1}"}
    )
    sequences = sequences.merge(m, on=f"id_{year}", how="inner")

for year in [2020, 2021, 2022, 2023]:
    crops = fields[year][["field_id", "hcat4_name"]].rename(
        columns={"field_id": f"id_{year}", "hcat4_name": f"crop_{year}"}
    )
    sequences = sequences.merge(crops, on=f"id_{year}", how="left")

sequences["field_id"] = range(1, len(sequences) + 1)

print("Complete sequences:", len(sequences))
display(sequences.head())

Complete sequences: 383496


,id_2020,id_2021,id_2022,id_2023,crop_2020,crop_2021,crop_2022,crop_2023,field_id
0,643580,623951,9679,551130,tree_wood_forest,tree_wood_forest,tree_wood_forest,tree_wood_forest,1
1,514540,661565,664340,563173,durum_hard_wheat,durum_hard_wheat,beans,unmaintained,2
2,203038,211257,196403,154310,not_known_and_other,not_known_and_other,not_known_and_other,not_known_and_other,3
3,202933,211184,192868,149748,not_known_and_other,not_known_and_other,not_known_and_other,not_known_and_other,4
4,661043,639769,641454,473129,not_known_and_other,not_known_and_other,not_known_and_other,not_known_and_other,5


In [15]:
sequences.to_csv("/kaggle/working/crop_rotations_2020_2023.csv", index=False)

from IPython.display import FileLink
display(FileLink("/kaggle/working/crop_rotations_2020_2023.csv"))

/kaggle/working/crop_rotations_2020_2023.csv

In [16]:
selected_crops = [
    'Alfalfa', 'Avocado', 'Avocado subtropical ecotype', 'Buckwheat',
    'Barley', 'Biomass sorghum', 'Cabbage', 'Camelina', 'Carrot',
    'Chickpea', 'Citrus', 'Cotton', 'Carinata', 'Finger millet',
    'Flax fibre', 'Foxtail millet', 'Groundnut', 'Miscanthus',
    'Maize', 'Silage maize', 'Oat', 'Olive', 'Onion', 'Dry pea',
    'Phaseolus bean', 'Pasture grasses', 'Reed canary grass',
    'Rapeseed', 'Rye', 'Sunflower', 'Solaris energy tobacco',
    'Soybean', 'Sorghum', 'Sugar beet', 'Switchgrass', 'Tef',
    'Tobacco', 'Tomato', 'Triticale', 'Wheat', 'Watermelon',
    'White potato'
]
print(sorted(crop_rotations_all["hcat4_name"].dropna().unique()))

['alfalfa_lucerne', 'almond', 'amaranth', 'apples', 'apricots', 'arable_crops', 'aromatic_medicinal_culinary_plants_spices_herbs', 'aronia_chokeberries', 'artemisia', 'artichoke', 'asparagus', 'aubergine_eggplant', 'avocado', 'barley', 'basil', 'beans', 'bell_pepper_paprika', 'beta_vulgaris', 'birch', 'birdsfoot_treifoil', 'blackberry', 'blackcurrant_cassis', 'borage', 'brassicaceae_cruciferae', 'broccoli', 'buckwheat', 'calendula_marigold', 'carrots_daucus', 'cauliflower', 'celeriac', 'celery', 'chamomile', 'cherry_cherries', 'chickpeas', 'chicory_chicories', 'chili_pepper', 'chives', 'chrysanthemum', 'citrus', 'clover', 'common_soft_wheat', 'conifers', 'coriander', 'cornflowers', 'crataegus_hawthorn', 'cucumber_pickle', 'dandelions', 'durum_hard_wheat', 'echinacea', 'elder_elderberry', 'endive', 'eucalyptus', 'fennel', 'fenugreek', 'festuca_fescue', 'fig', 'flax_linseed', 'flowers_ornamental_plants', 'fresh_vegetables', 'galega', 'garlic', 'gingko', 'greenhouse_foil_film', 'hazelnuts

In [17]:
from difflib import get_close_matches

available = crop_rotations_all["hcat4_name"].dropna().unique().tolist()

for crop in selected_crops:
    matches = get_close_matches(
        crop.lower().replace(" ", "_"),
        available,
        n=3,
        cutoff=0.4
    )
    print(f"{crop}: {matches}")

Alfalfa: ['alfalfa_lucerne', 'malva', 'galega']
Avocado: ['avocado', 'malva', 'tomato']
Avocado subtropical ecotype: ['cucumber_pickle', 'avocado']
Buckwheat: ['buckwheat', 'durum_hard_wheat', 'common_soft_wheat']
Barley: ['barley', 'parsley', 'basil']
Biomass sorghum: ['sorghum', 'asparagus', 'willows_osiers']
Cabbage: ['borage', 'orange', 'barley']
Camelina: ['scallion', 'phacelia', 'celeriac']
Carrot: ['yarrow', 'carrots_daucus', 'apricots']
Chickpea: ['chickpeas', 'echinacea', 'chives']
Citrus: ['citrus', 'other_citrus', 'iris']
Cotton: ['potatoes', 'scallion', 'conifers']
Carinata: ['nectarine', 'echinacea', 'valerian']
Finger millet: ['millet', 'cucumber_pickle', 'other_ribes']
Flax fibre: ['flax_linseed', 'festuca_fescue', 'olive']
Foxtail millet: ['millet', 'triticale', 'chamomile']
Groundnut: ['nuts', 'redcurrant', 'pomegranate']
Miscanthus: ['marian_thistles', 'amaranth', 'chrysanthemum']
Maize: ['millet', 'unmaintained', 'mandarin']
Silage maize: ['sage_chia', 'lilies', 'str

In [18]:
crop_mapping = {
    "maize_corn_popcorn": "Maize",
    "olive": "Olive",
    "sunflower": "Sunflower"
}

filtered_crops = crop_rotations_all[
    crop_rotations_all["hcat4_name"].isin(crop_mapping)
].copy()

filtered_crops["crop_name"] = filtered_crops["hcat4_name"].map(crop_mapping)

print("Original observations:", len(crop_rotations_all))
print("Filtered observations:", len(filtered_crops))

display(filtered_crops.head())

Original observations: 2592275
Filtered observations: 133292


,cropfield,original_code,off_id,off_area,area_ha,geometry,original_name,translated_name,hcat4_name,year,crop_name
7,290426,2402,182620,9290,0.928742,"MULTIPOLYGON (((11.514453 42.951141, 11.514382...",Oliva da trasformazione,Olive for processing,olive,2020,Olive
18,371730,2402,575899,674,0.067367,"MULTIPOLYGON (((11.492139 42.9021, 11.492087 4...",Oliva da trasformazione,Olive for processing,olive,2020,Olive
23,637621,2402,290229,1,0.000112,"MULTIPOLYGON (((11.389261 43.445416, 11.389221...",Oliva da trasformazione,Olive for processing,olive,2020,Olive
29,639876,2402,295033,129,0.012858,"MULTIPOLYGON (((11.937426 42.879653, 11.93711 ...",Oliva da trasformazione,Olive for processing,olive,2020,Olive
43,129608,2402,76298,1,0.000119,"MULTIPOLYGON (((11.18267 43.551746, 11.182656 ...",Oliva da trasformazione,Olive for processing,olive,2020,Olive


In [19]:
columns_to_drop = ["original_name", "translated_name", "hcat4_name"]

filtered_crops = filtered_crops.drop(columns=columns_to_drop, errors="ignore")

filtered_crops.to_csv(
    "/kaggle/working/eurocrops_selected_crops.csv",
    index=False
)

display(filtered_crops.head())

,cropfield,original_code,off_id,off_area,area_ha,geometry,year,crop_name
7,290426,2402,182620,9290,0.928742,"MULTIPOLYGON (((11.514453 42.951141, 11.514382...",2020,Olive
18,371730,2402,575899,674,0.067367,"MULTIPOLYGON (((11.492139 42.9021, 11.492087 4...",2020,Olive
23,637621,2402,290229,1,0.000112,"MULTIPOLYGON (((11.389261 43.445416, 11.389221...",2020,Olive
29,639876,2402,295033,129,0.012858,"MULTIPOLYGON (((11.937426 42.879653, 11.93711 ...",2020,Olive
43,129608,2402,76298,1,0.000119,"MULTIPOLYGON (((11.18267 43.551746, 11.182656 ...",2020,Olive


**geometry** - coordinates which describe the shape and the position of the cropfield

**cropfield** - id of the cropfield in the dataset 

**off_id** - original administrative id

**area_ha** - surface of the cropfield in hectares

**off_area** - original reported surface